# Exercise 2 — k-NN as an object, metrics, splitting, standardization

Last week you wrote a nearest-neighbour classifier as a script. Today we turn
it into a **model**: an object with `fit` and `predict`, measured with proper
metrics on data it has never seen, and fed with features that are on a
comparable scale. The data, the names `X` and `y` and the plotting helper are
the same as last week.

All definitions follow the second lecture — the same confusion matrix, the
same formulas for sensitivity, specificity, precision and $F_1$, the same
rule that scaling is fitted on the training part only.

Plan of the lab:

1. From NN to k-NN — the vote by hand, then the same in code.
2. OOP and the scikit-learn estimator API.
3. Metrics — first by hand, then in NumPy, then checked against scikit-learn.
4. Hold-out split.
5. Standardization, and why k-NN needs it.
6. Red flags — four broken pipelines to diagnose, the last one also to fix.
7. Homework: choosing $k$ on a validation set.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "https://raw.githubusercontent.com/tomasvicar/MLR-public/master/exercises/data/"

## Task 1 — From NN to k-NN

Last week the answer came from the *single* closest training sample. One
neighbour is a very noisy vote: a single mislabelled or unusual leaf next to
the new sample decides the whole prediction. **k-NN** looks at the $k$
closest training samples and takes a majority vote, which averages that noise
out — at the price of blurring the boundaries between classes when $k$ gets
large.

### 1a. The vote by hand

**Close the laptop for this part.** Here are the six leaves of last week's
by-hand task — two of each type — and a **seventh one, invented for this
exercise: an unusually large cherry that strayed among the walnuts**. The new
leaf is last week's as well, $h = 15$ cm and $w = 7$ cm.

Fill in the squared distance $d^2 = (h_i - 15)^2 + (w_i - 7)^2$ of every leaf
from the new one — as last week, $\sqrt{\cdot}$ is increasing, so comparing
$d^2$ is the same as comparing $d$, and all seven come out as whole numbers —
and then the rank of each leaf, 1 for the nearest.

<table border="1" style="border-collapse: collapse; text-align: center;">
<tr><th>#</th><th>Leaf height <i>h</i> [cm]</th><th>Leaf width <i>w</i> [cm]</th><th>Tree type</th><th style="width: 5em;"><i>d</i><sup>2</sup></th><th style="width: 5em;">rank</th></tr>
<tr><td>1</td><td>10</td><td>14</td><td>maple</td><td></td><td></td></tr>
<tr><td>2</td><td>14</td><td>16</td><td>maple</td><td></td><td></td></tr>
<tr><td>3</td><td>8</td><td>5</td><td>cherry</td><td></td><td></td></tr>
<tr><td>4</td><td>10</td><td>6</td><td>cherry</td><td></td><td></td></tr>
<tr><td>5</td><td>16</td><td>8</td><td>walnut</td><td></td><td></td></tr>
<tr><td>6</td><td>18</td><td>9</td><td>walnut</td><td></td><td></td></tr>
<tr><td>7</td><td>15</td><td>8</td><td>cherry</td><td></td><td></td></tr>
<tr><td><b>new</b></td><td><b>15</b></td><td><b>7</b></td><td><b>?</b></td><td></td><td></td></tr>
</table>

Then answer, still without a computer:

1. What does **1-NN** predict for the new leaf?
2. What does **3-NN** predict?
3. The two answers differ. Which of them would you trust, and why?

### 1b. The same vote in code

Now open the laptop and let the computer do it on all the leaves. The data is
last week's: 518 leaves described by height and width in centimetres, from a
maple, a cherry or a walnut tree. As in the first lab, `X` is the feature
matrix and `y` holds the tree types as strings.

In [ ]:
leaves = pd.read_csv(DATA + "leaves.csv")

X = leaves[["Leaf height", "Leaf width"]].to_numpy()
y = leaves["Tree type"].to_numpy()

new_sample = np.array([13.0, 6.0])  # a leaf we want to classify

print(X.shape, np.unique(y))

The same helper as last week, so the plots of the two labs can be compared.

In [ ]:
# one fixed colour per class, used in every plot below
COLORS = {"maple": "tab:green", "cherry": "tab:red", "walnut": "tab:brown"}


def plot_leaves(X, y, title):
    for tree_type in np.unique(y):
        rows = y == tree_type
        plt.scatter(X[rows, 0], X[rows, 1], c=COLORS.get(tree_type, "tab:blue"),
                    label=tree_type, alpha=0.6)
    plt.xlabel("Leaf height [cm]")
    plt.ylabel("Leaf width [cm]")
    plt.title(title)
    plt.legend(loc="lower right")


plot_leaves(X, y, "The leaves dataset")
plt.show()

Write the prediction as a **function**, so that we can call it with different
values of $k$ without copying the code. Three steps:

1. distance from the new sample to every training sample (`euk_dist` from
   last week, or one vectorised NumPy expression),
2. `np.argsort` on the distances gives the indices from the closest to the
   farthest — take the first $k$,
3. majority vote over the labels of those $k$ neighbours:
   `values, counts = np.unique(neighbour_labels, return_counts=True)`, then
   `values[np.argmax(counts)]`. The labels are strings, so nothing has to be
   encoded into numbers; `np.unique` sorts, so a tie is broken alphabetically.

In [ ]:
def euk_dist(a, b):
    return np.sqrt(np.sum((a - b) ** 2))


def knn_predict_one(X, y, sample, n_neighbors):
    """Class predicted for a single sample by majority vote of k neighbours."""
    # TODO: your code here
    return "maple"


prediction = knn_predict_one(X, y, new_sample, 5)
print("5-NN says:", prediction)

plot_leaves(X, y, "The new sample classified by 5-NN")
plt.scatter(new_sample[0], new_sample[1], c=COLORS[prediction], marker="X", s=250,
            edgecolors="black", linewidths=1.5, zorder=3, label=f"new sample -> {prediction}")
plt.legend(loc="upper left")
plt.show()

Now look at what $k$ does to one new sample, on all 518 leaves. Which class
does the nearest neighbour vote for, and which class wins once four more
neighbours join in? Expect the same flip you computed by hand in 1a — with a
different query leaf, $h = 13$, $w = 6$: the measurement $(15, 7)$ of 1a is
itself one of the 518 rows, so its nearest neighbour would sit at distance 0
and there would be nothing left to flip.

In [ ]:
distances = np.array([euk_dist(new_sample, X[i, :]) for i in range(X.shape[0])])
order = np.argsort(distances)

print("five nearest leaves:")
for i in order[:5]:
    print(f"  distance {distances[i]:5.2f}  {y[i]}")

print()
for k in [1, 3, 5, 7, 15]:
    print(f"k = {k:2d}  ->  {knn_predict_one(X, y, new_sample, k)}")

## Task 2 — OOP and the scikit-learn API

A function is enough for one prediction, but a model has **state**: the
training data, the hyperparameters, later the learned weights. That is what
objects are for. Python is an object-oriented language — almost everything in
it is an object with attributes and methods.

* **class** — the code that describes a type of object
* **object / instance** — one concrete variable of that class
* **attribute** — a variable stored inside the object
* **method** — a function that belongs to the object; its first argument
  `self` is the object itself

**Every model in scikit-learn is an object of exactly this kind**, and they all
speak the same language: hyperparameters go into the constructor, `fit(X, y)`
learns from the training data, `predict(X)` returns one label per row of `X`.
Because the API is identical for every model, swapping k-NN for a random forest
later in the semester is a one-line change. This is the line promised at the end
of the first lab.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X, y)                                   # string labels are fine
print(model.predict(new_sample.reshape(1, -1))[0])

Write `CustomKNeighborsClassifier` with the same behaviour — the same
constructor argument and the same two methods:

1. `fit(X, y)` only remembers the training data and returns `self`; k-NN has
   nothing else to learn, it is a *lazy* learner,
2. `predict(X)` takes a **matrix** of samples and returns one label per row,
   repeating the three steps of Task 1b for each of them.

In [ ]:
class CustomKNeighborsClassifier:
    def __init__(self, n_neighbors=3):
        self.n_neighbors = n_neighbors

    def fit(self, X, y):
        # TODO: your code here
        return self

    def predict(self, X):
        # TODO: your code here
        return np.array(["maple"] * X.shape[0])


model = CustomKNeighborsClassifier(n_neighbors=3)
model.fit(X, y)
print(model.predict(new_sample.reshape(1, -1))[0])

One prediction proves nothing. **Validate your implementation against
scikit-learn on the whole dataset** — this is the habit the whole course is
built on: whenever a reference implementation exists, compare against it
instead of trusting the output that "looks right".

Do not expect perfect agreement for every $k$. When two training samples sit
at *exactly* the same distance from the query, the $k$-th neighbour is
ambiguous and each implementation breaks the tie in its own order.

In [ ]:
for k in [1, 3, 5, 7]:
    pred_custom = CustomKNeighborsClassifier(n_neighbors=k).fit(X, y).predict(X)
    pred_sklearn = KNeighborsClassifier(n_neighbors=k).fit(X, y).predict(X)
    n_diff = int(np.sum(pred_custom != pred_sklearn))
    print(f"k = {k}: {n_diff} disagreement(s) out of {len(y)} samples")

## Task 3 — Metrics

### 3a. By hand, on the blackboard

**Close the laptop for this part.** A classifier decides for each of 100
leaves whether it is a maple (the *positive* class). Twenty of them really
are maples. The confusion matrix — rows are the actual class, columns the
prediction, as in the second lecture — is:

|                      | predicted maple | predicted not maple |
| -------------------- | --------------: | ------------------: |
| **actual maple**     |     $TP = 15$   |         $FN = 5$    |
| **actual not maple** |     $FP = 10$   |        $TN = 70$    |

Compute, as exact fractions and then as decimals:

$$\text{accuracy} = \frac{TP + TN}{TP + TN + FP + FN}, \qquad
  \text{sensitivity (recall)} = \frac{TP}{TP + FN}, \qquad
  \text{specificity} = \frac{TN}{TN + FP}$$

$$\text{precision} = \frac{TP}{TP + FP}, \qquad
  F_1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}}
                     {\text{precision} + \text{recall}}
      = \frac{2\,TP}{2\,TP + FP + FN}$$

One more, which Task 6 will report and which the lecture does not name:
**balanced accuracy** is the mean of the per-class recalls — the macro average
of the recall, in the lecture's terms:

$$\text{balanced accuracy} = \frac{1}{2}\left(\frac{TP}{TP + FN}
                                              + \frac{TN}{TN + FP}\right)$$

Compute it for the matrix above, and then for the two constant classifiers
"always maple" and "always not maple".

Then answer in one sentence: the accuracy looks good, so why would you not
want to report only that number?

### 3b. The same metrics in NumPy

Now the same formulas in code, on small vectors where you can still check the
result by counting on your fingers. Three steps — accuracy, then the binary
pair sensitivity/specificity, then the two regression errors.

**Step 1.** **Accuracy** on a four-class problem: no confusion matrix is
needed, accuracy is just the fraction of positions where the two vectors
agree. Expected result: 0.8.

In [ ]:
labels_cls = np.array([0, 1, 2, 2, 1, 0, 1, 2, 3, 3])
predictions_cls = np.array([0, 2, 1, 2, 1, 0, 1, 2, 3, 3])

# TODO: your code here
accuracy = ...

print("accuracy:", accuracy)

**Step 2.** **Sensitivity and specificity** are defined for a binary problem,
so we need the four counts first. A boolean array can be combined
element-wise with `&` (and) and `|` (or), and `np.sum` of a boolean array
counts the `True`s:

```python
both_true = (a == 1) & (b == 1)
```

Expected result: $TP = 5$, $FN = 2$, $FP = 1$, $TN = 2$.

In [ ]:
labels_bin = np.array([0, 1, 1, 1, 0, 0, 1, 1, 1, 1])
predictions_bin = np.array([0, 0, 1, 1, 0, 1, 1, 1, 1, 0])

# TODO: your code here
TP = FN = FP = TN = 0
sensitivity = ...
specificity = ...

print(f"TP = {TP}, FN = {FN}, FP = {FP}, TN = {TN}")
print("sensitivity:", sensitivity)
print("specificity:", specificity)

**Step 3.** For **regression** the labels are real numbers, so we measure how
far the prediction is, not whether it is right:

$$\mathrm{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2, \qquad
  \mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n}\lvert y_i - \hat{y}_i \rvert$$

MSE squares the errors, so one large mistake costs more than several small
ones; MAE treats every centimetre the same. Here `labels_reg` are the measured
leaf heights and `predictions_reg` come from a model that systematically
overestimates them. Watch the units: MAE is in centimetres, MSE in **square**
centimetres, so the two numbers are not directly comparable — but both come out
below one, and MSE below MAE, because every single error is below one
centimetre and squaring makes it smaller still.

In [ ]:
labels_reg = np.array([5.43, 8.64, 4.8, 9.32, 5.41, 2.73, 2.56, 8.68, 9.24, 7.95])
predictions_reg = np.array([6.35, 8.85, 5.66, 9.91, 6.36, 3.15, 2.76, 8.74, 9.94, 8.65])

# TODO: your code here
mse = ...
mae = ...

print("MSE:", mse)
print("MAE:", mae)

### 3c. Check everything against scikit-learn

Everything you just wrote already exists in `sklearn.metrics`. Run the cell
and compare — if a number differs, your NumPy version is wrong (or you are
computing a different metric than you think, which is just as common).

Note `pos_label` and `average`: for `recall_score` you have to say *which*
class is positive and how the per-class values are averaged. Getting these
arguments wrong is the most frequent silent bug in evaluation code.

One thing to watch in the printed matrix: `confusion_matrix` uses the same
orientation as the lecture (rows = actual, columns = predicted) but it **sorts
the class labels**, so with labels 0/1 the top-left cell is $TN$ and $TP$ is
bottom-right — the mirror image of the layout you filled in by hand in 3a. The
four counts are the same; only their position in the square differs.

In [ ]:
from sklearn.metrics import (accuracy_score, confusion_matrix, recall_score,
                             precision_score, f1_score,
                             mean_squared_error, mean_absolute_error, r2_score)

# the same three pairs of vectors you filled in above, nothing retyped
print("accuracy      :", accuracy_score(labels_cls, predictions_cls))
print("confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(labels_cls, predictions_cls))

print()
print("confusion matrix (labels sorted, so top-left is TN):")
print(confusion_matrix(labels_bin, predictions_bin))
print("sensitivity   :", recall_score(labels_bin, predictions_bin, pos_label=1))
print("specificity   :", recall_score(labels_bin, predictions_bin, pos_label=0))
print("precision     :", precision_score(labels_bin, predictions_bin, pos_label=1))
print("F1            :", f1_score(labels_bin, predictions_bin, pos_label=1))

print()
print("MSE           :", mean_squared_error(labels_reg, predictions_reg))
print("MAE           :", mean_absolute_error(labels_reg, predictions_reg))
print("R2            :", r2_score(labels_reg, predictions_reg))

$R^2 = 1 - \sum_i (y_i - \hat{y}_i)^2 / \sum_i (y_i - \bar{y})^2$ compares
the model with the constant "always predict the mean": $R^2 = 1$ is a perfect
fit, $R^2 = 0$ means no better than the mean, and a negative value means
worse than the mean. We will not implement it — the formula is in the second
lecture and `r2_score` is one line.

The same holds for the **ROC and precision–recall curves**: they belong to
the lecture, and in code they are `sklearn.metrics.roc_curve`,
`roc_auc_score` and `precision_recall_curve`.

## Task 4 — Hold-out split

So far every accuracy we computed was on the *training* data. For k-NN with
$k = 1$ that number is meaningless: every sample is its own nearest
neighbour, so training accuracy is 100 % for any dataset, however noisy —
unless two identical measurements carry different labels. That is the first of
last week's two closing questions, and red flag D below measures it.

The fix is the hold-out split from the second lecture: keep part of the data
aside, never let the model see it during training, and measure there. With a
hyperparameter to tune you need three parts — train to fit, validation to
choose, test to report once at the very end.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 1.6))
parts = [("Train\n70 %", 0.70, "#12355b"),
         ("Validation\n15 %", 0.15, "#1d5a7a"),
         ("Test\n15 %", 0.15, "#007f86")]
left = 0.0
for name, width, colour in parts:
    ax.barh(0, width, left=left, height=0.6, color=colour, edgecolor="white")
    ax.text(left + width / 2, 0, name, ha="center", va="center", color="white", fontsize=11)
    left += width
ax.set_xlim(0, 1)
ax.axis("off")
ax.set_title("fit the model   |   choose the hyperparameters   |   report once", fontsize=10)
plt.show()

Split the leaves 70 / 30, train k-NN with $k = 3$ on the training part and
measure accuracy on the testing part — twice, first by hand and then with
scikit-learn.

**Step 1.** In NumPy. `np.random.seed(42)` makes the shuffle reproducible —
without it you would get a different accuracy on every run and could not tell
an improvement from noise. `np.random.permutation(n)` returns the numbers
$0 \dots n-1$ in random order; use the first 70 % of them as training indices
and index the arrays with them (`X[train_idx, :]`).

Expected result: 362 training and 156 test samples.

In [ ]:
num_of_samples = X.shape[0]
np.random.seed(42)

# TODO: your code here
X_train, y_train, X_test, y_test = X, y, X, y
accuracy_numpy = 0.0

print(f"train {X_train.shape[0]} samples, test {X_test.shape[0]} samples")
print("test accuracy:", accuracy_numpy)

**Step 2.** scikit-learn does the same in one line. `stratify=y` keeps the
class proportions in both parts, which matters when a class is small — with a
purely random split you can end up with almost no cherries in the test set.
The two splits are different, so the two accuracies need not agree.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: your code here
X_train, X_test, y_train, y_test = X, X, y, y
accuracy_sklearn = 0.0

print(f"train {X_train.shape[0]} samples, test {X_test.shape[0]} samples")
print("test accuracy:", accuracy_sklearn)
print("difference against the NumPy split:", abs(accuracy_sklearn - accuracy_numpy))

One split gives one number, and that number is itself random — it depends on
which leaves happened to land in the test part. Before you believe that model
A is better than model B by one percentage point, look at how much a single
hold-out estimate moves when only the split changes.

In [ ]:
accuracies = []
for seed in range(10):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.3, random_state=seed, stratify=y)
    model = KNeighborsClassifier(n_neighbors=3).fit(X_tr, y_tr)
    accuracies.append(accuracy_score(y_te, model.predict(X_te)))

print("ten different splits of the same data:", np.round(accuracies, 4))
print(f"min {min(accuracies):.4f}, max {max(accuracies):.4f}, "
      f"spread {max(accuracies) - min(accuracies):.4f}")

## Task 5 — Standardization

k-NN measures Euclidean distance, and distance adds the features together.
A feature measured in tens of centimetres and a feature measured in tenths of
a centimetre therefore do **not** contribute equally — the large one decides
almost everything, no matter how informative the small one is.

The dataset for this task describes the same leaves by **leaf height** (units
of centimetres) and **stalk thickness** (tenths of a centimetre); they are
`X_stalk` and `y_stalk`. Both measurements are marked on the same leaf
below — the height of the blade, and the thickness of the stalk just under it.
The picture is where the factor of about 50 between the two features comes
from:

<img src="https://raw.githubusercontent.com/tomasvicar/MLR-public/master/exercises/data/leaf_measurement_stalk.png" width="220">

In [ ]:
stalk = pd.read_csv(DATA + "leaves_stalk.csv")

X_stalk = stalk[["Leaf height", "Stalk thickness"]].to_numpy()
y_stalk = stalk["Tree type"].to_numpy()

print(stalk.head())
print()
print("range of leaf height    :", X_stalk[:, 0].min(), "-", X_stalk[:, 0].max())
print("range of stalk thickness:", X_stalk[:, 1].min(), "-", X_stalk[:, 1].max())

The left panel below uses **equal aspect ratio**, so one centimetre on the
horizontal axis is one centimetre on the vertical axis — the same units the
Euclidean distance uses. That is what the classifier sees.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, title in zip(axes, ["as the distance sees it (equal aspect)",
                            "axes stretched to fit the data"]):
    plt.sca(ax)
    plot_leaves(X_stalk, y_stalk, title)
    plt.ylabel("Stalk thickness [cm]")

axes[0].set_aspect("equal", adjustable="box")
axes[0].set_xlim(0, 26)
axes[0].set_ylim(0, 26)
plt.tight_layout()
plt.show()

In the left panel the three classes lie on one horizontal line: the stalk
thickness is invisible. In the right panel — the same data, only the axis
scaling differs — the three classes are clearly separated *by the stalk
thickness*. Standardization is exactly what turns the left picture into the
right one for the classifier:

$$x' = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

Train k-NN with $k = 3$ twice, on the raw features and on the standardized
ones, and compare the test accuracy. Three steps:

1. mean and standard deviation of each column of the **training part only** —
   using the whole dataset would let information from the test data into the
   model (the leakage from the second lecture, and red flag A in Task 6),
2. subtract and divide, on the training part and on the test part alike,
3. fit k-NN on the standardized training part and score it on the
   standardized test part.

Expected result of step 1: mean $\approx [12.51,\ 0.22]$, standard deviation
$\approx [3.58,\ 0.09]$.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_stalk, y_stalk, test_size=0.3, random_state=42, stratify=y_stalk)

# k-NN on the raw features
model = KNeighborsClassifier(n_neighbors=3).fit(X_train, y_train)
accuracy_raw = accuracy_score(y_test, model.predict(X_test))

# TODO: your code here
mean, std = 0.0, 1.0
accuracy_std = 0.0

print("mean and std of the training part:", np.round(mean, 3), np.round(std, 3))
print(f"accuracy without standardization: {accuracy_raw:.4f}")
print(f"accuracy with standardization   : {accuracy_std:.4f}")

And the same with `StandardScaler`, which stores $\mu$ and $\sigma$ inside
the object: `fit_transform` on the training part, `transform` on the test
part. There is no `fit` on the test data — that is the whole point.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

model = KNeighborsClassifier(n_neighbors=3).fit(X_train_std, y_train)
print("scaler mean_ and scale_:", np.round(scaler.mean_, 3), np.round(scaler.scale_, 3))
print("accuracy with StandardScaler:", accuracy_score(y_test, model.predict(X_test_std)))

## Task 6 — Red flags

The rest of the semester you will read code you did not write, some of it
written by an agent. Code that runs, produces a plausible number and is
nevertheless wrong is the normal case, not the exception — and the mistake is
almost always in the *evaluation*, not in the model.

Four snippets follow, each in a code cell of its own. Each one runs without
an error and reports a number that is **too optimistic**. For each one: run
the snippet, read it, and write your diagnosis in the cell below it — what is
wrong and why the reported number is biased upwards. For the **first three**
you then run the cell *Measure red flag …*, which puts a number on the bias;
its code is collapsed on purpose, because reading it gives the answer away, and
in the practical test at the end of the semester there is no measurement to
read. The fourth snippet you repair yourself instead.

### Red flag A

Labelling is the expensive part of a dataset: this pilot study has only **20
labelled leaves** to train on, and the remaining 498 serve as the test set.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()                     # min-max normalization into [0, 1]
X_scaled = scaler.fit_transform(X_stalk)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_stalk, train_size=20,
                                                    random_state=42, stratify=y_stalk)
model = KNeighborsClassifier(n_neighbors=3).fit(X_train, y_train)
print("accuracy:", round(accuracy_score(y_test, model.predict(X_test)), 4))

What did the model learn that it should not have? Answer first — the collapsed
cell then measures the bias on the split the snippet used and, because a single
split of 20 training leaves is noisy, as an average over 200 such splits.

*Your diagnosis of red flag A:*

In [ ]:
#@title Measure red flag A — run it only after you wrote your diagnosis
X_scaled = MinMaxScaler().fit_transform(X_stalk)      # the leaky transform, all 518 leaves


def leaky_and_correct(seed):
    """Accuracy of the leaky and of the correct pipeline on one and the same split."""
    Xtr, Xte, ytr, yte = train_test_split(X_stalk, y_stalk, train_size=20,
                                          random_state=seed, stratify=y_stalk)
    Xtr_leak, Xte_leak, _, _ = train_test_split(X_scaled, y_stalk, train_size=20,
                                                random_state=seed, stratify=y_stalk)
    leaky = accuracy_score(yte, KNeighborsClassifier(n_neighbors=3)
                           .fit(Xtr_leak, ytr).predict(Xte_leak))
    sc = MinMaxScaler().fit(Xtr)                      # the training part only
    correct = accuracy_score(yte, KNeighborsClassifier(n_neighbors=3)
                             .fit(sc.transform(Xtr), ytr).predict(sc.transform(Xte)))
    return leaky, correct


X_fit_20, _, _, _ = train_test_split(X_stalk, y_stalk, train_size=20,
                                     random_state=42, stratify=y_stalk)
mm_all, mm_train = MinMaxScaler().fit(X_stalk), MinMaxScaler().fit(X_fit_20)
print("min, max from all 518 leaves      :", mm_all.data_min_, mm_all.data_max_)
print("min, max from the 20 training ones:", mm_train.data_min_, mm_train.data_max_)

leaky, correct = leaky_and_correct(42)
print(f"\nthe split the snippet used: leaky {leaky:.4f}, correct {correct:.4f}, "
      f"bias {leaky - correct:+.4f}")

runs = np.array([leaky_and_correct(seed) for seed in range(200)])
print(f"averaged over 200 splits:  leaky {runs[:, 0].mean():.4f}, "
      f"correct {runs[:, 1].mean():.4f}, bias {runs[:, 0].mean() - runs[:, 1].mean():+.4f}")
print(f"the leaky pipeline scored higher in {int(np.sum(runs[:, 0] > runs[:, 1]))} "
      f"of the 200 splits and lower in {int(np.sum(runs[:, 0] < runs[:, 1]))}")

### Red flag B

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_stalk, y_stalk, test_size=0.2,
                                                    random_state=42, stratify=y_stalk)
scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

ks = list(range(1, 32, 2))
scores = []
for k in ks:
    model = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)
    scores.append(accuracy_score(y_test, model.predict(X_test_s)))
print("best accuracy:", round(max(scores), 4))

The code uses a proper train/test split and even fits the scaler on the
training part only. So why is the printed number still not an estimate of the
performance on new data?

*Your diagnosis of red flag B:*

In [ ]:
#@title Measure red flag B — run it only after you wrote your diagnosis
# the honest way: k is chosen on a validation part cut off the training data
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25,
                                              random_state=42, stratify=y_train)
scaler_fit = StandardScaler().fit(X_fit)
val_scores = [accuracy_score(y_val, KNeighborsClassifier(n_neighbors=k)
                             .fit(scaler_fit.transform(X_fit), y_fit)
                             .predict(scaler_fit.transform(X_val))) for k in ks]

k_from_test = ks[int(np.argmax(scores))]
k_from_val = ks[int(np.argmax(val_scores))]
# the chosen k is refitted on the whole training part - the same model as in the snippet
honest = scores[ks.index(k_from_val)]

print(f"k picked on the TEST set:       k = {k_from_test}, reported accuracy {max(scores):.4f}")
print(f"k picked on the VALIDATION set: k = {k_from_val}, honest test accuracy {honest:.4f}")
print(f"optimistic bias: {max(scores) - honest:+.4f}")

plt.plot(ks, val_scores, "o-", label="validation")
plt.plot(ks, scores, "s-", label="test")
plt.xlabel("k")
plt.ylabel("accuracy")
plt.legend()
plt.show()

### Red flag C

Here the task is binary: *is this leaf a maple?*, decided from the leaf
height alone, on a sample of leaves in which 95 % really are maples. The
target is therefore 0/1 rather than a name, because a binary metric has to be
told which class is the positive one.

In [ ]:
rng = np.random.default_rng(1)
idx = np.concatenate([
    rng.choice(np.where(y == "maple")[0], 190, replace=False),
    rng.choice(np.where(y == "cherry")[0], 5, replace=False),
    rng.choice(np.where(y == "walnut")[0], 5, replace=False)])
idx = np.sort(idx)

X_imb = X[idx, :1]                          # leaf height only
y_imb = (y[idx] == "maple").astype(int)     # 1 = maple, 0 = not maple
print(f"{len(y_imb)} leaves, {100 * y_imb.mean():.0f} % of them maples")

X_train, X_test, y_train, y_test = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb)

model = KNeighborsClassifier(n_neighbors=3).fit(X_train, y_train)
print("accuracy:", round(accuracy_score(y_test, model.predict(X_test)), 4))

An accuracy of 0.95 — great? What would you need to know before you believe
that number?

*Your diagnosis of red flag C:*

In [ ]:
#@title Measure red flag C — run it only after you wrote your diagnosis
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score

pred_knn = KNeighborsClassifier(n_neighbors=3).fit(X_train, y_train).predict(X_test)
# the dumbest baseline there is: always the most frequent training class
pred_baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train).predict(X_test)

for name, pred in [("k-NN, k = 3", pred_knn),
                   ('DummyClassifier("most_frequent")', pred_baseline)]:
    print()
    print(name)
    print(f"  accuracy           {accuracy_score(y_test, pred):.4f}")
    print(f"  balanced accuracy  {balanced_accuracy_score(y_test, pred):.4f}")
    print(f"  recall (maple)     {recall_score(y_test, pred, pos_label=1):.4f}")
    print(f"  recall (not maple) {recall_score(y_test, pred, pos_label=0):.4f}")
    print(f"  macro F1           {f1_score(y_test, pred, average='macro'):.4f}")
    print(f"  micro F1           {f1_score(y_test, pred, average='micro'):.4f}")
    print("  confusion matrix (rows = actual, columns = predicted):")
    print("   ", confusion_matrix(y_test, pred).tolist())

### Red flag D

In [ ]:
model = KNeighborsClassifier(n_neighbors=1).fit(X, y)
print("accuracy:", round(accuracy_score(y, model.predict(X)), 4))

This one prints the best number of the four. Write your diagnosis below.

*Your diagnosis of red flag D:*

Now **repair the snippet** in the cell below. Three steps:

1. split 70 / 30 with `train_test_split`, `random_state=42`, `stratify=y`,
2. fit the same $k = 1$ model on the training part only,
3. score it on the test part.

Expected result: 362 / 156 samples and a test accuracy around 0.92 — not 0.99.

In [ ]:
# TODO: your code here
X_tr, X_te, y_tr, y_te = X, X, y, y
honest = 0.0

print(f"train {X_tr.shape[0]} samples, test {X_te.shape[0]} samples")
print(f"honest test accuracy:          {honest:.4f}")

## Task 7 (homework) — choosing $k$ on a validation set

$k$ is a hyperparameter: it is not learned from the training data, we have to
pick it. Picking the value that gives the best *test* accuracy would make the
test set part of the training procedure, and the reported number would be too
optimistic (that is red flag B in Task 6). So we need a third part of the data.

The three-way split (60 / 20 / 20) and the scaler fitted on the training part
are already written for you in the cell below — they are the two steps you did
in Tasks 4 and 5. Your part is what is new:

1. Loop over $k = 1, 2, \dots, 30$, train on the training part and evaluate on
   the **validation** part. Plot the accuracy against $k$.
2. Take the best $k$ and report its accuracy on the **testing** part — once.
3. Answer in three sentences: is the validation accuracy of the best $k$ a fair
   estimate of how the model will do on new leaves? Why? Which of the two
   numbers would you put into a report?

Expected result of the split: 310 training, 104 validation and 104 testing
samples.

In [ ]:
# given: the three-way split and the scaler fitted on the training part only
X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5, random_state=42, stratify=y_rest)

scaler = StandardScaler().fit(X_train)
X_train_s, X_val_s, X_test_s = (scaler.transform(X_train), scaler.transform(X_val),
                                scaler.transform(X_test))
print(f"{len(y_train)} train, {len(y_val)} validation, {len(y_test)} test samples")

ks = np.arange(1, 31)

# TODO: your code here
...

*Your three-sentence answer:*

## Summary

* k-NN votes among $k$ neighbours; $k$ is a hyperparameter, not a parameter.
* A model is an object: `__init__` takes the hyperparameters, `fit` learns,
  `predict` returns one label per row.
* Whenever a reference implementation exists, compare against it.
* Accuracy alone hides everything that matters on unbalanced data.
* Fit the scaler on the training part; choose hyperparameters on the
  validation part; touch the test part once — and never report a number
  measured on the training data.

Two questions to answer before you leave — the first of them comes back in
the next lab:

- In Task 5 one of the two features carries almost all the information. Which
  one — and how would you decide that from the data alone, without training a
  classifier on each of them?
- Standardizing *both* features lifted the accuracy from 0.81 to 0.97, but
  standardizing the leaf height alone leaves a height-only k-NN exactly where
  it was. Why can rescaling a single feature never change what k-NN answers?